In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Region.csv"
region_uncleaned = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", "\t")
    .load(path))

In [0]:
from pyspark.sql import functions as F
region_uncleaned.select("Country").distinct().show()

region_country_std = region_uncleaned.withColumn(
    "Country",
    F.when(F.col("Country").isin("United States","USA"), "United States")
     .otherwise(F.col("Country"))
)

region_country_std.select("Country").distinct().show()

### Checking if after name standarization, we have any duplicates. (no dups)

In [0]:
total_rows = region_country_std.count()
unique_rows = region_country_std.dropDuplicates().count()

duplicate_count = total_rows - unique_rows
print("Total duplicated rows:", duplicate_count)

In [0]:
column_mapping = {
    "SalesTerritoryKey": "sales_territory_key",
    "Region": "region",
    "Country": "country",
    "Group": "group"
}

In [0]:
region_silver = region_country_std
for old_name, new_name in column_mapping.items():
    region_silver = region_silver.withColumnRenamed(old_name, new_name)
    print(f"Renamed: {old_name} → {new_name}")
region_silver.printSchema()
display(region_silver)

In [0]:
region_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_lakehouse.silver.region")

In [0]:
%sql
DROP TABLE IF EXISTS sales_lakehouse.silver.region

In [0]:
%sql
SELECT * FROM sales_lakehouse.silver.region;